# Reinforcement Learning

# 3. Online evaluation

This notebook presents the online evaluation of a policy by **Monte-Carlo learning** and **TD learning**.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
from model import Maze, Walk, TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineEvaluation
from dynamic import PolicyEvaluation

## To do

* Complete the class ``MCLearning`` and test it on the random walk.
* Compare with the exact solution obtained by Dynamic Programming.<br> You might plot the [Spearman's correlation](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient) between both value functions with respect to the training time.
* Do the same with ``TDLearning``.

## Monte-Carlo learning

In [ ]:
class MCLearning(OnlineEvaluation):
    """Online evaluation by Monte-Carlo."""
        
    def update_values(self, state=None, horizon=100):
        """Update the values from one episode."""
        stop, states, rewards = self.get_episode(state=state, horizon=horizon)
        states.pop()
        gain = 0
        # backward update
        for state, reward in zip(reversed(states), reversed(rewards)):
            self.add_state(state)
            code = self.model.encode(state)
            self.count[code] += 1
            # to be modified
            # begin
            gain = reward + self.gamma * gain
            # end 
            diff = gain - self.value[code]
            count = self.count[code]
            self.value[code] += diff / count
            

## TD learning

In [ ]:
class TDLearning(OnlineEvaluation):
    """Online evaluation by TD learning."""
        
    def update_values(self, state=None, horizon=100):
        """Update values online from one episode."""
        self.model.reset(state)
        for t in range(horizon):
            state = self.model.state
            self.add_state(state)
            code = self.model.encode(state)
            stop = self.model.is_terminal(state)
            
            if stop:
                break

            action = self.get_action(state)
            reward, stop = self.model.step(action)
            
            next_state = self.model.state
            self.add_state(next_state)
            next_code = self.model.encode(next_state)

            self.count[code] += 1
            gain = reward + self.gamma * self.value[next_code]
            
            diff = gain - self.value[code]
            count = self.count[code]
            self.value[code] += diff / count


### Walk

#### MC learning

In [ ]:
walk = Walk()
algo_mc = MCLearning(walk, policy='random', gamma=0.9)

n_episodes = 10
for t in range(n_episodes):
    algo_mc.update_values()

policy_mc = algo_mc.get_policy()
values_mc = algo_mc.get_values()

walk.display_policy(policy_mc)
walk.display_values(values_mc)

#### TD learning

In [ ]:
walk = Walk()
algo_td = TDLearning(walk, policy='random', gamma=0.9)

horizon = 1000
algo_td.update_values(horizon=horizon)

policy_td = algo_td.get_policy()
values_td = algo_td.get_values()

walk.display_policy(policy_td)
walk.display_values(values_td)

#### Policy iteration

In [ ]:
walk = Walk()
algo_pe = PolicyEvaluation(walk, policy='random', gamma=0.9)

algo_pe.evaluate_policy()

values_pe = algo_pe.values
policy_pe = algo_pe.get_policy()

walk.display_policy(policy_pe)
walk.display_values(values_pe)

### Comparison

In [ ]:
walk = Walk()
algo_mc = MCLearning(walk, policy='random', gamma=0.9)
correlations_mc = []

n_episodes = 100
step = 1
for t in range(n_episodes+1):
    algo_mc.update_values()
    if t%step == 0:
        values = algo_mc.get_values()
        corr = stats.spearmanr(values, values_pe)[0] if t > 0 else 0
        correlations_mc.append(corr)

In [ ]:
walk = Walk()
algo = TDLearning(walk, policy='random', gamma=0.9)
correlations_td = []

n_episodes = 100
state = walk.init_state()
for t in range(n_episodes+1):
    algo.update_values(state=state)
    
    values = algo.get_values()
    corr = stats.spearmanr(values, values_pe)[0] if t > 0 else 0
    correlations_td.append(corr)

    state = algo.model.state # continue from the last state (one continuous episode)

In [ ]:
# Plot episode vs correlation
episodes = np.arange(0, n_episodes+1, step)

plt.plot(episodes, correlations_mc, label='MC')
plt.plot(episodes, correlations_td, label='TD')
plt.xlabel('Episode')
plt.ylabel('Correlation')
plt.ylim(-0.03, 1)
plt.legend()
plt.show()

## To do

Test the other environments:
* The maze: can you find the exit after policy improvement?<br> You might adapt the number of episodes used for training.
* The games (Tic-Tac-Toe, Nim, Connect Four): can you beat a random player after policy improvement? a player with the one-step policy?<br> Comment the results.

## Maze

In [ ]:
maze_map = np.load('maze.npy')

init_state = (1, 0)
exit_state = (1, 20)
Maze.set_parameters(maze_map, init_state, [exit_state])

maze = Maze()
maze.display()

In [ ]:
algo_mc = MCLearning(maze, policy='random')

n_episodes = 2000
for t in range(n_episodes):
    algo_mc.update_values(state='random')

policy_mc = algo_mc.get_policy()
values_mc = algo_mc.get_values()

maze.display_policy(policy_mc)
maze.display_values(values_mc)

In [ ]:
maze = Maze()
algo_td = TDLearning(maze, policy='random')

n_episodes = 2000
for t in range(n_episodes):
    algo_td.update_values(state='random')

policy_td = algo_td.get_policy()
values_td = algo_td.get_values()

maze.display_policy(policy_td)
maze.display_values(values_td)

**Comments on maze performance**

No, the agent after policy improvement cannot find the exit. The agent is capable of finding the exit only when its in a state somewhat near the exit. It still struggles on deep parts of the maze where it gets stuck in a loop between 2 states. This is the case for both MC and TD learning.

## Games

In [ ]:
def run_training(Game, Algo, adversary_policy='random', n_episodes=1000, verbose=False):
    """Run the training of a given agent on a given game."""
    game = Game(adversary_policy=adversary_policy)
    algo = Algo(game, policy='random')

    for t in range(n_episodes):
        algo.update_values()

    policy_improved = algo.get_policy()
    agent = Agent(game, policy=policy_improved)
        
    results = dict(zip(*np.unique(agent.get_gains(), return_counts=True)))

    if verbose:
        print("Game: {}".format(Game.__name__))
        print("Adversary: {}".format(adversary_policy))
        print("Learning algorithm: {}".format(Algo.__name__), end='\n\n')

        print("Losses: {}".format(results[-1] if -1 in results else 0))
        print("Draws: {}".format(results[0] if 0 in results else 0))
        print("Wins: {}".format(results[1] if 1 in results else 0))
        
    return results

### TicTacToe

#### Random adversary

In [ ]:
n_episodes = 10000

In [ ]:
results = run_training(TicTacToe, MCLearning, 'random', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(TicTacToe, TDLearning, 'random', n_episodes=n_episodes, verbose=True)

#### One-step policy adversary

In [ ]:
results = run_training(TicTacToe, MCLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(TicTacToe, TDLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

### Nim

In [ ]:
n_episodes = 10000

#### Random adversary

In [ ]:
results = run_training(Nim, MCLearning, adversary_policy='random', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(Nim, TDLearning, adversary_policy='random', n_episodes=n_episodes, verbose=True)

#### One-step policy adversary

In [ ]:
results = run_training(Nim, MCLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(Nim, TDLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

### ConnectFour

In [ ]:
n_episodes = 1000

#### Random adversary

In [ ]:
results = run_training(ConnectFour, MCLearning, adversary_policy='random', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(ConnectFour, TDLearning, adversary_policy='random', n_episodes=n_episodes, verbose=True)

#### One-step policy adversary

In [ ]:
results = run_training(ConnectFour, MCLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

In [ ]:
results = run_training(ConnectFour, TDLearning, adversary_policy='one_step', n_episodes=n_episodes, verbose=True)

**Comments on games results**

On TicTacToe, the improved policy agent can win most of the times (+95%) against a random policy adversary. However, it wins ~50% of the times and ties the other ~50% against a one-step policy adversary.

On Nim, the improved agent wins most of the times (85-95%) against both random and one-step policy players.

On ConnectFour, the improved policy agent has a +80% win rate against a random policy player and losses most of the times (~75%) against a one-step policy adversary.

The improved policy is much better than a random policy player for simple games (TicTacToe and Nim), but it is not even close to being a perfect player. It is however, sometimes better than a one-step policy player (e.g. Nim). One interpretation is that when the state space becomes larger, MC learning and TD learning would need a very large amount of iterations in order to get close to the true value function. This would explain why this learning algorithms work on the random walk but struggle on the maze and some of the games (TicTacToe against one-step policy and ConnectFour).